In [ ]:
# ============================================================
# THEIA — TRAINING (DET + POSE) SAVED IN GOOGLE DRIVE
# ============================================================
# - Mounts Drive and sets persistent folders
# - Verifies/loads dataset (ZIP in Drive or already extracted folder)
# - Derives datasets:
#     * yolo_det_dataset  (detection with bounding boxes)
#     * yolo_pose_crops   (crops + keypoints kpt_shape=[32,3])
# - Trains:
#     * YOLOv8-nano (detect) on full images
#     * YOLOv8-medium (pose) on crops
# - Saves to Drive:
#     * runs/detect/train_nano (weights, metrics, plots)
#     * runs/pose/train_medium (weights, metrics, plots)
#     * derived datasets (optional, recommended)
#     * MANIFEST.json with experiment summary
# ============================================================

import os, sys, time, json, glob, shutil, zipfile
from pathlib import Path

# ---------- 0) DRIVE MOUNT AND PERSISTENT PATHS ----------
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT       = "/content/drive/MyDrive"
PROJECT_FOLDER   = "Theia_Project_Data"               # USER MUST ADJUST THIS FOLDER NAME TO THEIR GOOGLE DRIVE PATH
PROJECT_ROOT     = f"{DRIVE_ROOT}/{PROJECT_FOLDER}"                    # persistent
DATASET_ZIP      = f"{PROJECT_ROOT}/yolo_dataset.zip"                  # if dataset is zipped
DATASET_DIR_DR   = f"{PROJECT_ROOT}/yolo_dataset"                      # persistent dataset (optional)
DATASET_DIR_RAM  = "/content/yolo_dataset"                             # RAM workspace for faster I/O

# Derived datasets (also saved in Drive)
DET_ROOT_DR      = f"{PROJECT_ROOT}/yolo_det_dataset"
POSE_ROOT_DR     = f"{PROJECT_ROOT}/yolo_pose_crops"

# Derived in RAM (generated here and copied to Drive later)
DET_ROOT_RAM     = "/content/yolo_det_dataset"
POSE_ROOT_RAM    = "/content/yolo_pose_crops"

# Training runs output DIRECTLY to Drive to prevent data loss
RUNS_DET_DIR     = f"{PROJECT_ROOT}/runs/detect"
RUNS_POSE_DIR    = f"{PROJECT_ROOT}/runs/pose"

# Create base target directories in Drive
for d in [PROJECT_ROOT, RUNS_DET_DIR, RUNS_POSE_DIR]:
    os.makedirs(d, exist_ok=True)

print("[PATHS]")
print("  Project:", PROJECT_ROOT)
print("  Dataset ZIP:", DATASET_ZIP)
print("  Dataset RAM:", DATASET_DIR_RAM)
print("  DET runs  :", RUNS_DET_DIR)
print("  POSE runs :", RUNS_POSE_DIR)

# ---------- 1) MINIMAL DEPENDENCIES INSTALLATION ----------
import subprocess
def pip_install(pkgs): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

# Stable Ultralytics release + basic dependencies
pip_install(["ultralytics==8.3.226", "opencv-python-headless", "pillow", "rich"])

import torch
from rich import print
print(f"[CHECK] Torch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

# ---------- 2) DATASET LOAD / EXTRACTION ----------
def ensure_dataset():
    # Priority 1: If dataset already exists in RAM (/content), use it:
    if os.path.exists(DATASET_DIR_RAM) and os.path.isdir(DATASET_DIR_RAM):
        print("[DATASET] Using dataset in RAM:", DATASET_DIR_RAM)
        return DATASET_DIR_RAM

    # Priority 2: If dataset exists in Drive as a folder, copy it to RAM (faster)
    if os.path.exists(DATASET_DIR_DR) and os.path.isdir(DATASET_DIR_DR):
        print("[DATASET] Copying dataset from Drive to RAM...")
        shutil.copytree(DATASET_DIR_DR, DATASET_DIR_RAM)
        return DATASET_DIR_RAM

    # Priority 3: If ZIP exists in Drive, extract to RAM and optionally save copy in Drive
    if os.path.exists(DATASET_ZIP):
        print("[DATASET] Extracting ZIP from Drive to RAM...")
        with zipfile.ZipFile(DATASET_ZIP, "r") as z:
            z.extractall("/content")
        # If ZIP contained a yolo_dataset folder, it will be at /content/yolo_dataset
        if not os.path.exists(DATASET_DIR_RAM):
            # Edge case: Zip contained an intermediate folder; detect and move it
            cands = glob.glob("/content/**/images", recursive=True)
            for c in cands:
                if c.endswith("yolo_dataset/images"):
                    base = str(Path(c).parents[1])
                    shutil.move(base, DATASET_DIR_RAM)
                    break
        # Optional: Persist dataset copy in Drive if missing
        if not os.path.exists(DATASET_DIR_DR):
            print("[DATASET] Saving copy to Drive...")
            shutil.copytree(DATASET_DIR_RAM, DATASET_DIR_DR)
        return DATASET_DIR_RAM

    raise FileNotFoundError(
        "Dataset not found. Please upload 'yolo_dataset.zip' to "
        f"{PROJECT_ROOT} o crea la carpeta {DATASET_DIR_DR} con 'images' y 'labels' (train/val)."
    )

DATASET_DIR = ensure_dataset()

# Quick diagnostic check
def count_pairs(split):
    imgs = glob.glob(f"{DATASET_DIR}/images/{split}/*")
    lbls = glob.glob(f"{DATASET_DIR}/labels/{split}/*.txt")
    return len(imgs), len(lbls)
tr_i, tr_l = count_pairs("train")
va_i, va_l = count_pairs("val")
print(f"[DATASET] train imgs={tr_i} labels={tr_l} | val imgs={va_i} labels={va_l}")

# ---------- 3) DERIVE DET AND CROPS FOR POSE ----------
import math
import numpy as np
from PIL import Image

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".JPG", ".JPEG", ".PNG"}
PADDING_RATIO = 0.15  # padding ratio around the bbox for the crop

def list_images(folder):
    return [p for p in glob.glob(os.path.join(folder, "*")) if os.path.splitext(p)[1] in IMG_EXTS]

def label_path_for(img_path, labels_root):
    stem = os.path.splitext(os.path.basename(img_path))[0]
    return os.path.join(labels_root, stem + ".txt")

def parse_pose_label_line(line):
    vals = [float(x) for x in line.strip().split()]
    cls  = int(vals[0])
    cx, cy, w, h = vals[1:5]
    kpts = vals[5:]
    if len(kpts) != 64:
        raise ValueError(f"Expected 64 values (32 points x 2), but got {len(kpts)}")
    return cls, cx, cy, w, h, kpts

def clamp01(v): return max(0.0, min(1.0, float(v)))

def safe_image_open(path):
    try:
        im = Image.open(path); im.load(); return im
    except Exception:
        return None

def clean_split(images_dir, labels_dir):
    removed = {"img_no_label":0, "lbl_no_img":0, "oob":0, "bad_img":0}
    imgs = list_images(images_dir)
    for img_path in imgs:
        lbl_path = label_path_for(img_path, labels_dir)
        if not os.path.exists(lbl_path):
            os.remove(img_path); removed["img_no_label"] += 1; continue
        im = safe_image_open(img_path)
        if im is None:
            os.remove(img_path); os.remove(lbl_path); removed["bad_img"] += 1; continue
        with open(lbl_path, "r") as f:
            lines = [ln.strip() for ln in f if ln.strip()]
        keep = True
        for ln in lines:
            try:
                cls,cx,cy,w,h,kpts = parse_pose_label_line(ln)
            except Exception:
                keep = False; break
            nums = [cx,cy,w,h] + kpts
            if not all(0.0 <= float(v) <= 1.0 for v in map(float, nums)):
                keep = False; break
        if not keep:
            os.remove(img_path); os.remove(lbl_path); removed["oob"] += 1
    # Handle orphan labels (without an associated image)
    for lbl_path in glob.glob(os.path.join(labels_dir, "*.txt")):
        stem = os.path.splitext(os.path.basename(lbl_path))[0]
        if not any(os.path.exists(os.path.join(images_dir, stem + ext)) for ext in IMG_EXTS):
            os.remove(lbl_path); removed["lbl_no_img"] += 1
    return removed

# Cleaning
for split in ["train", "val"]:
    print(f"[CLEAN] {split}")
    stats = clean_split(f"{DATASET_DIR}/images/{split}", f"{DATASET_DIR}/labels/{split}")
    print("   Removed ->", stats)

# Setup RAM directories
for d in [DET_ROOT_RAM, POSE_ROOT_RAM]:
    for sub in ["images/train","images/val","labels/train","labels/val"]:
        os.makedirs(f"{d}/{sub}", exist_ok=True)

# 3.a) Derive DET dataset from pose labels
for split in ["train","val"]:
    for img_path in list_images(f"{DATASET_DIR}/images/{split}"):
        lbl_pose = label_path_for(img_path, f"{DATASET_DIR}/labels/{split}")
        if not os.path.exists(lbl_pose): continue
        out_img = f"{DET_ROOT_RAM}/images/{split}/{os.path.basename(img_path)}"
        if not os.path.exists(out_img):
            shutil.copy2(img_path, out_img)
        with open(lbl_pose, "r") as f:
            lines = [ln.strip() for ln in f if ln.strip()]
        det_lines = []
        for ln in lines:
            cls,cx,cy,w,h,kpts = parse_pose_label_line(ln)
            det_lines.append(f"{cls} {cx} {cy} {w} {h}")
        with open(f"{DET_ROOT_RAM}/labels/{split}/{Path(img_path).stem}.txt","w") as f:
            f.write("\n".join(det_lines))

with open(f"{DET_ROOT_RAM}/data_det.yaml","w") as f:
    f.write(f"path: {DET_ROOT_RAM}\ntrain: images/train\nval: images/val\nnames: ['flower']\n")

print("[OK] DET Dataset in RAM:", DET_ROOT_RAM)

# 3.b) Generate crops for POSE and remap keypoints kpt_shape=[32,3] (x,y,v=2)
def crop_and_relabel(img_path, label_path, out_img_dir, out_lbl_dir, pad_ratio=0.15):
    im = Image.open(img_path).convert("RGB")
    W, H = im.size
    with open(label_path, "r") as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    kept = 0
    for idx, ln in enumerate(lines):
        cls,cx,cy,w,h,kpts = parse_pose_label_line(ln)
        bw, bh = w*W, h*H
        bx, by = cx*W, cy*H
        x1, y1 = bx - bw/2, by - bh/2
        x2, y2 = bx + bw/2, by + bh/2
        pad_w = bw*pad_ratio; pad_h = bh*pad_ratio
        x1p = max(0, int(x1 - pad_w))
        y1p = max(0, int(y1 - pad_h))
        x2p = min(W, int(x2 + pad_w))
        y2p = min(H, int(y2 + pad_h))
        crop_w = max(1, x2p - x1p)
        crop_h = max(1, y2p - y1p)
        crop = im.crop((x1p, y1p, x2p, y2p))
        out_img_name = f"{Path(img_path).stem}_obj{idx}.jpg"
        crop.save(os.path.join(out_img_dir, out_img_name), quality=95)

        kxy = np.array(kpts, dtype=np.float32).reshape(-1,2)
        kx_abs = kxy[:,0] * W
        ky_abs = kxy[:,1] * H
        kx_rel = (kx_abs - x1p) / crop_w
        ky_rel = (ky_abs - y1p) / crop_h
        out_k = []
        for x_,y_ in zip(kx_rel, ky_rel):
            out_k += [clamp01(x_), clamp01(y_), 2]  # v=2 denotes visible keypoint

        cx_new = clamp01((bx - x1p) / crop_w)
        cy_new = clamp01((by - y1p) / crop_h)
        w_new  = clamp01(bw / crop_w)
        h_new  = clamp01(bh / crop_h)

        with open(os.path.join(out_lbl_dir, f"{Path(out_img_name).stem}.txt"), "w") as f:
            f.write("0 " + " ".join([
                f"{cx_new:.6f}", f"{cy_new:.6f}", f"{w_new:.6f}", f"{h_new:.6f}",
                *[f"{v:.6f}" if isinstance(v, float) else str(v) for v in out_k]
            ]))
        kept += 1
    return kept

kept_summary = {}
for split in ["train","val"]:
    in_img_dir = f"{DATASET_DIR}/images/{split}"
    in_lbl_dir = f"{DATASET_DIR}/labels/{split}"
    out_img_dir = f"{POSE_ROOT_RAM}/images/{split}"
    out_lbl_dir = f"{POSE_ROOT_RAM}/labels/{split}"
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lbl_dir, exist_ok=True)
    kept = 0
    for img_path in list_images(in_img_dir):
        lbl_path = label_path_for(img_path, in_lbl_dir)
        if not os.path.exists(lbl_path): continue
        try:
            kept += crop_and_relabel(img_path, lbl_path, out_img_dir, out_lbl_dir, pad_ratio=PADDING_RATIO)
        except Exception as e:
            pass
    kept_summary[split] = kept
    print(f"[CROPS] {split}: {kept} objects/crops generated.")

with open(f"{POSE_ROOT_RAM}/data_pose.yaml","w") as f:
    f.write(
        f"path: {POSE_ROOT_RAM}\ntrain: images/train\nval: images/val\n"
        "kpt_shape: [32, 3]\n# flip_idx: []\n"
        "names: ['flower']\n"
    )

print("[OK] POSE Dataset (crops) in RAM:", POSE_ROOT_RAM)

# ---------- 4) BACKUP DERIVED DATASETS TO DRIVE (PERSISTENT) ----------
def rsync_dir(src, dst):
    os.makedirs(dst, exist_ok=True)
    # recursive copy preserving structure; using shutil in Colab instead of !rsync
    if os.path.exists(dst):
        # clear target (optional; comment out for incremental syncing)
        pass
    if os.path.abspath(src) != os.path.abspath(dst):
        if os.path.exists(dst):
            # copy file-by-file to allow re-running without complete erasure
            for root, _, files in os.walk(src):
                rel = os.path.relpath(root, src)
                tgt = os.path.join(dst, rel)
                os.makedirs(tgt, exist_ok=True)
                for f in files:
                    s = os.path.join(root, f)
                    d = os.path.join(tgt, f)
                    if not os.path.exists(d):
                        shutil.copy2(s, d)

print("[SAVE] Copying derived datasets to Drive (only the first time is slow) ...")
rsync_dir(DET_ROOT_RAM,  DET_ROOT_DR)
rsync_dir(POSE_ROOT_RAM, POSE_ROOT_DR)
print("[SAVE] Derived datasets saved in Drive:")
print("  -", DET_ROOT_DR)
print("  -", POSE_ROOT_DR)

# ---------- 5) TRAINING (outputs straight to DRIVE) ----------
from ultralytics import YOLO
IMG_SIZE    = 640
DET_EPOCHS  = 120
POSE_EPOCHS = 160
BATCH_DET   = 16
BATCH_POSE  = 16
WORKERS     = 2
DEVICE      = 0  # Target GPU 0 in Colab

det_yaml  = f"{DET_ROOT_RAM}/data_det.yaml"
pose_yaml = f"{POSE_ROOT_RAM}/data_pose.yaml"

assert os.path.exists(det_yaml),  f"Missing {det_yaml} (please derive datasets first)."
assert os.path.exists(pose_yaml), f"Missing {pose_yaml} (please derive datasets first)."

t0 = time.time()

# 5.a) Nano detector (detection)
det_model = YOLO("yolov8n.pt")  # initialize from pre-trained weights
det_res = det_model.train(
    task="detect",
    data=det_yaml,
    imgsz=IMG_SIZE,
    epochs=DET_EPOCHS,
    batch=BATCH_DET,
    device=DEVICE,
    workers=WORKERS,
    project=RUNS_DET_DIR,     # <-- training logs/weights persist in Drive
    name="train_nano",
    exist_ok=True,
    verbose=True,
)
print("[INFO] DET run dir:", det_res.save_dir)

# 5.b) Medium pose (on image crops)
pose_model = YOLO("yolov8m-pose.pt")  # initialize from pre-trained weights
pose_res = pose_model.train(
    task="pose",
    data=pose_yaml,
    imgsz=IMG_SIZE,
    epochs=POSE_EPOCHS,
    batch=BATCH_POSE,
    device=DEVICE,
    workers=WORKERS,
    project=RUNS_POSE_DIR,    # <-- training logs/weights persist in Drive
    name="train_medium",
    exist_ok=True,
    verbose=True,
)
print("[INFO] POSE run dir:", pose_res.save_dir)

print(f"[THEIA] Total training duration: {(time.time()-t0)/3600:.2f} h")

# ---------- 6) EXPERIMENT LOG MANIFEST ----------
import pandas as pd

def recent_results_csv(run_dir):
    csvs = sorted(glob.glob(f"{run_dir}/results*.csv"))
    return csvs[-1] if csvs else None

manifest = {
  "project_root": PROJECT_ROOT,
  "datasets": {
    "source": DATASET_DIR,
    "det_drive": DET_ROOT_DR,
    "pose_drive": POSE_ROOT_DR,
    "crops_generated": kept_summary,
  },
  "runs": {
    "det": {
      "run_dir": det_res.save_dir,
      "weights_best": f"{det_res.save_dir}/weights/best.pt",
      "weights_last": f"{det_res.save_dir}/weights/last.pt",
      "results_csv": recent_results_csv(det_res.save_dir),
    },
    "pose": {
      "run_dir": pose_res.save_dir,
      "weights_best": f"{pose_res.save_dir}/weights/best.pt",
      "weights_last": f"{pose_res.save_dir}/weights/last.pt",
      "results_csv": recent_results_csv(pose_res.save_dir),
    }
  },
  "train_params": {
    "img_size": IMG_SIZE,
    "det_epochs": DET_EPOCHS,
    "pose_epochs": POSE_EPOCHS,
    "batch_det": BATCH_DET,
    "batch_pose": BATCH_POSE,
    "workers": WORKERS,
    "device": DEVICE,
    "kpt_shape": [32,3],
    "padding_ratio_crops": PADDING_RATIO
  },
  "env": {
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
  },
  "timestamp_end": time.strftime("%Y-%m-%d_%H-%M-%S")
}

with open(f"{PROJECT_ROOT}/MANIFEST.json", "w") as f:
    json.dump(manifest, f, indent=2)
print("[SAVE] MANIFEST.json written to:", f"{PROJECT_ROOT}/MANIFEST.json")

# Print brief summary
for k, v in manifest["runs"].items():
    print(f"[{k.upper()}] best.pt:", v["weights_best"])
    if v["results_csv"]:
        try:
            df = pd.read_csv(v["results_csv"])
            print(f"  recent metric rows ({k}):")
            display(df.tail(1))
        except Exception:
            pass